# 01 — Walmart Retail Sales Analysis & Forecasting

**Objective:** Explore Walmart weekly sales, prepare store/feature data, train baseline regression models, and compare them using a chronological holdout.

## Expected files
Place these files in `data/walmart/`:
- `train.csv` (required; includes `Store`, `Dept`, `Date`, `Weekly_Sales`, `IsHoliday`)
- `features.csv` (required; includes `Store`, `Date`, `IsHoliday`)
- `stores.csv` (required; includes `Store`)

This notebook uses the original Walmart-style dataset schema. Check your own dataset's column names before running. It does not contain precomputed results.

In [ ]:
# 1. Imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
DATA_DIR = Path("data/walmart")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

pd.set_option("display.max_columns", 100)

In [ ]:
# 2. Load data and validate required columns
required_files = ["train.csv", "features.csv", "stores.csv"]
missing_files = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing {missing_files} in {DATA_DIR.resolve()}. "
        "Download the authorized dataset and place the files in data/walmart/."
    )

sales = pd.read_csv(DATA_DIR / "train.csv")
features = pd.read_csv(DATA_DIR / "features.csv")
stores = pd.read_csv(DATA_DIR / "stores.csv")

required_sales = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_features = {"Store", "Date", "IsHoliday"}
assert required_sales.issubset(sales.columns), f"Missing sales columns: {required_sales - set(sales.columns)}"
assert required_features.issubset(features.columns), f"Missing feature columns: {required_features - set(features.columns)}"
assert "Store" in stores.columns, "stores.csv must contain Store"

print("Sales:", sales.shape, "Features:", features.shape, "Stores:", stores.shape)
display(sales.head())

In [ ]:
# 3. Merge and basic data understanding
df = sales.merge(features, on=["Store", "Date", "IsHoliday"], how="left")
df = df.merge(stores, on="Store", how="left", suffixes=("", "_store"))
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date", "Weekly_Sales"]).copy()
df = df.sort_values("Date").reset_index(drop=True)

print("Merged shape:", df.shape)
display(df.describe(include="all").T.head(20))
print("Missing values (top 15):")
display(df.isna().sum().sort_values(ascending=False).head(15))

In [ ]:
# 4. Exploratory analysis
weekly = df.groupby("Date", as_index=False)["Weekly_Sales"].sum()
plt.figure(figsize=(13, 4))
plt.plot(weekly["Date"], weekly["Weekly_Sales"])
plt.title("Total Walmart Weekly Sales")
plt.xlabel("Date"); plt.ylabel("Weekly sales"); plt.grid(alpha=.25); plt.show()

if "Type" in df.columns:
    display(df.groupby("Type")["Weekly_Sales"].agg(["mean", "median", "count"]))

In [ ]:
# 5. Feature engineering
# Date-derived features are safe for forecasting. No future sales values are used.
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfYear"] = df["Date"].dt.dayofyear
df["IsHoliday"] = df["IsHoliday"].astype(int)

# Optional markdown fields: missing values represent no markdown in many releases.
markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
for col in markdown_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Store/department IDs and categorical metadata are predictors, not targets.
candidate_features = [
    c for c in ["Store", "Dept", "IsHoliday", "Year", "Month", "WeekOfYear",
                "DayOfYear", "Temperature", "Fuel_Price", "CPI", "Unemployment",
                "Type", "Size"] + markdown_cols if c in df.columns
]
X = df[candidate_features].copy()
y = df["Weekly_Sales"].astype(float)

categorical = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric = [c for c in X.columns if c not in categorical]
print("Features:", candidate_features)

In [ ]:
# 6. Chronological train/test split
# Hold out the latest 20% of unique dates to avoid randomly mixing future periods into training.
dates = np.sort(df["Date"].unique())
cutoff = dates[max(1, int(len(dates) * 0.8))]
train_mask = df["Date"] < cutoff
test_mask = ~train_mask

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
print("Cutoff:", cutoff, "| Train:", X_train.shape, "| Test:", X_test.shape)
if len(X_test) == 0 or len(X_train) == 0:
    raise ValueError("Chronological split produced an empty partition; inspect date coverage.")

In [ ]:
# 7. Preprocessing and model training
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )
}

# XGBoost is optional; skip cleanly if it is not installed.
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1
    )
except ImportError:
    print("XGBoost not installed; skipping. Install with: pip install xgboost")

fitted = {}
records = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    metrics = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    }
    fitted[name] = pipe
    print(name, metrics)
    for metric, value in metrics.items():
        records.append({"Dataset": "Walmart", "Model": name, "Metric": metric, "Value": float(value)})

In [ ]:
# 8. Save measured results and predictions
walmart_results = pd.DataFrame(records)
display(walmart_results)
walmart_results.to_csv(RESULTS_DIR / "walmart_model_results.csv", index=False)

best_name = min(
    walmart_results[walmart_results["Metric"] == "RMSE"]["Model"],
    key=lambda n: walmart_results.query("Model == @n and Metric == 'RMSE'")["Value"].iloc[0]
)
best_model = fitted[best_name]
predictions = pd.DataFrame({
    "Actual_Weekly_Sales": y_test.to_numpy(),
    "Predicted_Weekly_Sales": best_model.predict(X_test)
})
predictions.to_csv(RESULTS_DIR / "walmart_test_predictions.csv", index=False)
print("Lowest test RMSE model:", best_name)

In [ ]:
# 9. Visualize actual vs predicted
plt.figure(figsize=(6, 6))
plt.scatter(predictions["Actual_Weekly_Sales"], predictions["Predicted_Weekly_Sales"], alpha=.25)
lims = [
    min(predictions.min()),
    max(predictions.max())
]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual weekly sales"); plt.ylabel("Predicted weekly sales")
plt.title(f"Actual vs Predicted — {best_name}"); plt.grid(alpha=.2); plt.show()

## Interpretation checklist

Before publishing:
- Confirm the split is chronological and matches your forecasting objective.
- Record the date cutoff and any filtering/sampling.
- Interpret MAE/RMSE in the original sales units.
- Do not claim generalization beyond the held-out period.
- Save the notebook after running it so outputs reflect your actual experiment.